# Dataset preprocessing
We will use this script to clean out real-world dataset

In [1]:
import os
import json
import pickle
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import cv2
from scipy.spatial.transform import Rotation as R

### Setting sensor parameters

In [2]:
# Path to dataset folder
DATASET_PATH = "./dataset/real_data2"

# Load Camera Parameters
with open(os.path.join(DATASET_PATH, "cam_param.json"), "r") as f:
    cam_params = json.load(f)

# Load LiDAR Parameters
with open(os.path.join(DATASET_PATH, "lidar_param.json"), "r") as f:
    lidar_params = json.load(f)

# Load IMU Parameters
with open(os.path.join(DATASET_PATH, "imu_param.json"), "r") as f:
    imu_params = json.load(f)

### Loading input data
Inputs: 
1. cmd vel - calculated from pose
2. pose: pose.txt 
3. motor_speed - wheel_speed of all 4 wheels - wheel_speed.txt
4. dt - calculated from timestamp difference - timestamp.txt
5. pose_diff - calculated from pose
6. time - timestamp.txt

In [3]:
# Load Motion Data
pose_data = np.loadtxt(os.path.join(DATASET_PATH, "pose.txt"))  
# vel_data = np.loadtxt(os.path.join(DATASET_PATH, "vel.txt"))  
wheel_speed = np.loadtxt(os.path.join(DATASET_PATH, "wheel_speed.txt"))
timestamps = np.loadtxt(os.path.join(DATASET_PATH, "timestamp.txt")) 

Convert quaternion pose to euclidean pose

In [4]:
pos = pose_data[:, 0:3]  # assuming x, y, z
quat = pose_data[:, 3:]  # assuming x, y, z, qx, qy, qz, qw
rpy = R.from_quat(quat).as_euler('xyz', degrees=False)  # now shape is (N, 3)
pose_data = np.hstack((pose_data[:, 0:3], rpy))  # final shape: (N, 6)
print(pose_data.shape)  # (N, 6)    

(1314, 6)


Obtain terrain patches - footprint/ paths

In [5]:
def load_footprint_images(data_dir):
    DATASET_DIR = Path(data_dir)
    FOOTPRINT_DIR = DATASET_DIR / "footprint"

    # Create footprint directory if it doesn't exist
    FOOTPRINT_DIR.mkdir(exist_ok=True)

    # Load RGB Images (Convert to 40x40 Grayscale)
    image_paths = sorted(glob(os.path.join(data_dir, "cam/rgb/*.png")))  # Load RGB images
    footprint_images = []
    for idx, img_path in tqdm(enumerate(image_paths), total=len(image_paths)):
        img = Image.open(img_path).convert("L")  # Convert to grayscale
        img = img.resize((40, 40))  # Resize to 40x40
        img = np.array(img)
        footprint_images.append(img)
        
        npy_filename = FOOTPRINT_DIR / f"mocap_{idx}.npy"
        np.save(npy_filename, img)

    footprint_images = np.array(footprint_images)
    return footprint_images

In [6]:
def load_footprint_paths(data_dir):
    DATASET_DIR = Path(data_dir)
    FOOTPRINT_DIR = DATASET_DIR / "footprint"

    image_paths = os.listdir(FOOTPRINT_DIR)
    image_paths = sorted(image_paths, key=(lambda x: int(x.split('_')[-1].split('.')[0])))
    print(image_paths)

    footprint_paths = [ f'./footprint/{path}' for path in image_paths ]
    # footprint_paths = sorted(footprint_paths, key=(lambda x: x.split('_')[-1]))

    return footprint_paths


footprint_images = load_footprint_images(DATASET_PATH)
footprints = load_footprint_paths(DATASET_PATH)

100%|██████████| 1314/1314 [00:13<00:00, 100.74it/s]

['mocap_0.npy', 'mocap_1.npy', 'mocap_2.npy', 'mocap_3.npy', 'mocap_4.npy', 'mocap_5.npy', 'mocap_6.npy', 'mocap_7.npy', 'mocap_8.npy', 'mocap_9.npy', 'mocap_10.npy', 'mocap_11.npy', 'mocap_12.npy', 'mocap_13.npy', 'mocap_14.npy', 'mocap_15.npy', 'mocap_16.npy', 'mocap_17.npy', 'mocap_18.npy', 'mocap_19.npy', 'mocap_20.npy', 'mocap_21.npy', 'mocap_22.npy', 'mocap_23.npy', 'mocap_24.npy', 'mocap_25.npy', 'mocap_26.npy', 'mocap_27.npy', 'mocap_28.npy', 'mocap_29.npy', 'mocap_30.npy', 'mocap_31.npy', 'mocap_32.npy', 'mocap_33.npy', 'mocap_34.npy', 'mocap_35.npy', 'mocap_36.npy', 'mocap_37.npy', 'mocap_38.npy', 'mocap_39.npy', 'mocap_40.npy', 'mocap_41.npy', 'mocap_42.npy', 'mocap_43.npy', 'mocap_44.npy', 'mocap_45.npy', 'mocap_46.npy', 'mocap_47.npy', 'mocap_48.npy', 'mocap_49.npy', 'mocap_50.npy', 'mocap_51.npy', 'mocap_52.npy', 'mocap_53.npy', 'mocap_54.npy', 'mocap_55.npy', 'mocap_56.npy', 'mocap_57.npy', 'mocap_58.npy', 'mocap_59.npy', 'mocap_60.npy', 'mocap_61.npy', 'mocap_62.npy', '

Calculating command velocity from pose

In [7]:
# Compute Time Differences
dt_values = np.diff(timestamps, prepend=timestamps[0])

# Compute Pose Differences
pose_diff = np.diff(pose_data, axis=0, prepend=pose_data[0:1, :])

In [8]:
dp = np.diff(pos, axis=0)
lin_vel = np.linalg.norm(dp, axis=1) / dt_values[1:]  # velocity in m/s
lin_vel = np.insert(lin_vel, 0, 0)  # Insert 0 for the first element

In [9]:
def quat_conjugate(q):
    """Return quaternion conjugate of [qx,qy,qz,qw]."""
    return np.array([-q[0], -q[1], -q[2], q[3]])

def quat_multiply(q2, q1):
    """Return Hamilton product q2 * q1."""
    x1, y1, z1, w1 = q1
    x2, y2, z2, w2 = q2
    w = w2*w1 - x2*x1 - y2*y1 - z2*z1
    x = w2*x1 + x2*w1 + y2*z1 - z2*y1
    y = w2*y1 - x2*z1 + y2*w1 + z2*x1
    z = w2*z1 + x2*y1 - y2*x1 + z2*w1
    return np.array([x, y, z, w])

In [10]:
def calc_angular(quats, dt):
    rel_quats = []
    for i in range(len(quats)-1):
        q1 = quats[i]
        q2 = quats[i+1]
        q_rel = quat_multiply(q2, quat_conjugate(q1))
        # normalize
        q_rel = q_rel / np.linalg.norm(q_rel)
        rel_quats.append(q_rel)
    rel_quats = np.stack(rel_quats, axis=0)            # (N-1, 4)

    # Angle of rotation: θ = 2 * arccos(q_rel_w)
    angles = 2 * np.arccos(np.clip(rel_quats[:, 3], -1.0, 1.0))  # (N-1,)
    omega = angles / dt 
    return omega

ang_vel = calc_angular(quat, dt_values[1:])
ang_vel = np.insert(ang_vel, 0, 0)  # Insert 0 for the first element

In [11]:
print('Length of dataset 1')
print(len(lin_vel), len(ang_vel), len(timestamps), len(pose_data), len(footprints))

Length of dataset 1
1314 1314 1314 1314 1314


In [12]:
cmd_vel = np.vstack((lin_vel, ang_vel)).T  # (N, 2)
print(cmd_vel.shape)  # (N, 2)

(1314, 2)


### Compile all data

In [13]:
# Compute Time Differences
dt_values = np.diff(timestamps, prepend=timestamps[0])

# Compute Pose Differences
pose_diff = np.diff(pose_data, axis=0, prepend=pose_data[0:1, :])

# Create Dataset Dictionary
dataset_name = DATASET_PATH.split('/')[-1]
print(dataset_name)
dataset = {
    "bag_name": [dataset_name],
    "data": []
}

# Structure Data for Training
assert len(footprints) == len(cmd_vel) == len(pose_data) == len(wheel_speed) == len(dt_values) == len(pose_diff) == len(timestamps), "Mismatch in data lengths"
num_samples =  len(footprints)  # Ensure matching sizes

sample = {
    "cmd_vel": cmd_vel,  # (v, ω)
    "footprint": footprints,  # 40x40 grayscale image
    "pose": pose_data,  # (x, y, z, roll, pitch, yaw)
    "motor_speed": wheel_speed,  # If missing, use zeros
    "dt": np.array(dt_values),  # Time difference
    "pose_diff": pose_diff,  # Pose changes
    "time": np.array(timestamps)  # Timestamp
}

dataset["data"].append(sample)

real_data2


In [14]:
# Save Processed Data
with open("./vertiencoder/data/train/data_total.pkl", "wb") as f:
    pickle.dump(dataset, f)

print("✅ Dataset saved as data_total.pkl with", num_samples, "samples.")


✅ Dataset saved as data_total.pkl with 1314 samples.


### Creating training split

In [15]:
train_dataset = {
    "bag_name": dataset["bag_name"],
    "data":  [{}]
}

TRAIN_SPLIT = int(0.8 * num_samples)

for key in dataset['data'][0].keys():
    train_dataset['data'][0][key] = dataset['data'][0][key][:TRAIN_SPLIT]

In [16]:
len(train_dataset['data'][0]['footprint'])

1051

To have correct footprints assigned to correct train or validation folders, it's better to copy all the footprints twice, Once in training & other in validation folders. 

In [17]:
# Save Processed Data
with open("./vertiencoder/data/train/data_train.pkl", "wb") as f:
    pickle.dump(train_dataset, f)

In [18]:
# from shutil import copy

# SOURCE_PATH = './dataset/test_data1/footprint/'
# footprints = os.listdir(SOURCE_PATH)
# DEST_PATH = './vertiencoder/data/train/footprint/'

# TRAIN_SPLIT = int(0.8 * len(footprints))
# train_footprints = footprints[:TRAIN_SPLIT]
# val_footprints = footprints[TRAIN_SPLIT:]

# for patch in train_footprints:
#     src_path = os.path.join(SOURCE_PATH, patch)
#     copy(src_path, DEST_PATH)

In [19]:
# DEST_PATH = './vertiencoder/data/val/footprint/'

# for patch in val_footprints:
#     src_path = os.path.join(SOURCE_PATH, patch)
#     copy(src_path, DEST_PATH)

### Creating validation dataset

In [20]:
valid_dataset = {
    "bag_name": dataset["bag_name"],
    "data":  [{}]
}

for key in dataset['data'][0].keys():
    valid_dataset['data'][0][key] = dataset['data'][0][key][TRAIN_SPLIT:]

In [21]:
len(valid_dataset['data'][0]['footprint'])

263

In [22]:
with open('./vertiencoder/data/val/data_val.pkl', 'wb') as f:
    pickle.dump(valid_dataset, f)

### Running stats script

In [25]:
! python ./vertiencoder/utils/stats.py

name: train
cmd_vel_mean: tensor([19.2525,  0.2937])
cmd_vel_var: tensor([89.6961,  7.3612])
cmd_vel_std: tensor([9.4708, 2.7132])
cmd_vel_max: tensor([52.1982, 62.3624])
cmd_vel_min: tensor([0., 0.])
footprint_mean: 0.7404950857162476
footprint_var: 0.45033949613571167
footprint_std: 0.6710733771324158
footprint_max: 1.0
footprint_min: -1.0
pose_mean: tensor([ 4.5997e+01, -2.9602e+01,  1.2360e+02,  3.1027e-03, -1.0196e-03,
        -5.6645e-01])
pose_var: tensor([7.8069e+04, 3.3121e+04, 3.9128e+01, 2.1287e-03, 5.1905e-04, 4.0568e+00])
pose_std: tensor([2.7941e+02, 1.8199e+02, 6.2553e+00, 4.6138e-02, 2.2783e-02, 2.0142e+00])
pose_max: tensor([3.8223e+02, 2.3642e+02, 1.3446e+02, 1.9093e-01, 6.1341e-02, 3.1415e+00])
pose_min: tensor([-3.9848e+02, -2.8945e+02,  1.1492e+02, -9.7624e-02, -1.9924e-01,
        -3.1414e+00])
motor_speed_mean: tensor([19.2529, 19.1675, 19.4730, 19.4458])
motor_speed_var: tensor([87.0514, 86.1709, 87.4692, 86.1326])
motor_speed_std: tensor([9.3301, 9.2828, 9.3525